# Третье задание NLP: русские эмбеддинги и retrieval

**Датасет:** Анар — «Шестой этаж пятиэтажного дома» (FB2)

Структура ноутбука:
- **Часть 0** — Установка и импорты
- **Часть 1** — Готовые русские эмбеддинги (word2vec-ruscorpora-300)
- **Часть 2** — Обучение Word2Vec на своём датасете
- **Часть 3** — Построение retrieval-индекса

---
## Часть 0: Установка и импорты

In [13]:
!pip install gensim faiss-cpu numpy scikit-learn

In [14]:
import re
import numpy as np
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import faiss
import warnings
warnings.filterwarnings('ignore')

print(f"gensim version: {gensim.__version__}")
print(f"faiss version: {faiss.__version__}")

gensim version: 4.4.0
faiss version: 1.13.2


---
## Часть 1: Готовые русские эмбеддинги

### Загрузка модели

Используем предобученную модель  из gensim-data.

Эта модель обучена на Национальном корпусе русского языка (ruscorpora) и использует формат , где POS — часть речи (NOUN, VERB, ADJ и т.д.).

In [15]:
# Загружаем предобученную модель (может занять несколько минут)
model_ru = api.load('word2vec-ruscorpora-300')
print(f"Размер словаря: {len(model_ru.key_to_index)}")
print(f"Размерность вектора: {model_ru.vector_size}")

Размер словаря: 184973
Размерность вектора: 300


### Анализ препроцессинга

Модель ruscorpora использует формат . Посмотрим на примеры ключей в словаре:

In [16]:
# Посмотрим на формат ключей в модели
sample_keys = list(model_ru.key_to_index.keys())[:30]
print("Примеры ключей в словаре:")
for k in sample_keys:
    print(f"  {k}")

Примеры ключей в словаре:
  весь_DET
  человек_NOUN
  мочь_VERB
  год_NOUN
  сказать_VERB
  время_NOUN
  говорить_VERB
  становиться_VERB
  знать_VERB
  самый_DET
  дело_NOUN
  день_NOUN
  жизнь_NOUN
  рука_NOUN
  очень_ADV
  первый_ADJ
  давать_VERB
  новый_ADJ
  слово_NOUN
  иметь_VERB
  большой_ADJ
  идти_VERB
  глаз_NOUN
  место_NOUN
  лицо_NOUN
  видеть_VERB
  хотеть_VERB
  понимать_VERB
  должный_ADJ
  работа_NOUN


### Вывод о препроцессинге

Для работы с этой моделью необходимо:
1. **Лемматизировать** слово (привести к начальной форме)
2. **Добавить POS-тег** через нижнее подчёркивание (например, , )
3. Слова записываются в нижнем регистре

Это означает, что простой запрос слова без POS-тега **не найдёт** его в модели. Нужно знать часть речи.

### Выбор слов и ближайшие соседи

Выбираем 15 слов из разных доменов: повседневная жизнь, природа, абстрактные понятия, эмоции, действия.

In [17]:
model_ru = api.load('word2vec-ruscorpora-300')
print(f"Размер словаря: {len(model_ru.key_to_index)}")
print(f"Размерность вектора: {model_ru.vector_size}")

# Список слов с POS-тегами для поиска в модели ruscorpora
words_with_pos = [
    # Повседневная жизнь / люди
    ("человек_NOUN", "человек"),
    ("дом_NOUN", "дом"),
    ("город_NOUN", "город"),
    ("мать_NOUN", "мать"),
    ("друг_NOUN", "друг"),
    # Природа / мир
    ("море_NOUN", "море"),
    ("солнце_NOUN", "солнце"),
    ("дерево_NOUN", "дерево"),
    # Абстрактные понятия
    ("любовь_NOUN", "любовь"),
    ("время_NOUN", "время"),
    ("счастье_NOUN", "счастье"),
    ("смерть_NOUN", "смерть"),
    # Действия
    ("идти_VERB", "идти"),
    ("думать_VERB", "думать"),
    # Прилагательные
    ("большой_ADJ", "большой"),
]

print("Ближайшие соседи в модели word2vec-ruscorpora-300:\n")

for word_pos, word_display in words_with_pos:
    if word_pos in model_ru.key_to_index:
        neighbors = model_ru.most_similar(word_pos, topn=10)
        neighbor_str = ", ".join([f"{w.split('_')[0]} ({s:.3f})" for w, s in neighbors])
        print(f"🔹 {word_display} ({word_pos}):")
        print(f"   {neighbor_str}\n")
    else:
        print(f"❌ {word_display} ({word_pos}) — не найдено в словаре\n")

Размер словаря: 184973
Размерность вектора: 300
Ближайшие соседи в модели word2vec-ruscorpora-300:

🔹 человек (человек_NOUN):
   женщина (0.550), мужчина (0.516), человеческий (0.501), идолопоклонствовать (0.484), высокопорядочный (0.482), правдознатец (0.482), некорыстолюбивый (0.480), народ (0.477), старое::стариться (0.475), людишки (0.474)

🔹 дом (дом_NOUN):
   дома (0.684), домик (0.672), особняк (0.668), квартира (0.636), флигель (0.633), двухэтажный (0.599), полукаменный (0.586), мезонин (0.574), усадьба (0.573), изба (0.571)

🔹 город (город_NOUN):
   столица (0.753), городок (0.676), пригород (0.666), змеиногорск (0.590), алейск (0.579), городишко (0.577), житель (0.577), предместье (0.577), москва (0.576), городской (0.568)

🔹 мать (мать_NOUN):
   отец (0.738), бабушка (0.687), сестра (0.686), тетка (0.676), дочь (0.668), родитель (0.660), муж (0.659), жена (0.645), сын (0.643), мама (0.643)

🔹 друг (друг_NOUN):
   дружка (0.747), взаимно (0.643), единомыслие::исповем (0.599),

### Наблюдения по Части 1

1. **Семантическая близость работает хорошо**: для слова «море» находятся «океан», «побережье», «берег» — семантически связанные слова.
2. **POS-тег фильтрует часть речи**: соседи существительных — преимущественно существительные, глаголов — глаголы. Это следствие формата .
3. **Абстрактные понятия** (любовь, счастье, смерть) имеют более «размытых» соседей, что отражает их многозначность в языке.
4. **Необходимость лемматизации**: без приведения к начальной форме и добавления POS-тега поиск невозможен — это ключевая особенность препроцессинга данной модели.

---
## Часть 2: Обучение Word2Vec на своём датасете

### Загрузка и подготовка данных

Используем книгу Анара «Шестой этаж пятиэтажного дома» в формате FB2.

In [18]:
# Загрузка и парсинг FB2
FB2_PATH = "/content/Анар . Шестой этаж пятиэтажного дома - royallib.ru.fb2"

with open(FB2_PATH, 'rb') as f:
    raw = f.read()

text = raw.decode('windows-1251')

# Извлекаем параграфы из FB2
paragraphs_raw = re.findall(r'<p>(.*?)</p>', text, re.DOTALL)
paragraphs = [re.sub(r'<[^>]+>', '', p).strip() for p in paragraphs_raw]
paragraphs = [p for p in paragraphs if len(p) > 20]  # убираем короткие служебные строки

print(f"Всего абзацев: {len(paragraphs)}")
print(f"Общее количество символов: {sum(len(p) for p in paragraphs)}")
print("\nПример абзаца:")
print(f"{paragraphs[5][:300]}...")

Всего абзацев: 1308
Общее количество символов: 323146

Пример абзаца:
Так и есть — вслед за физически ощутимой гулкой тишиной наступил и успокоительный мрак. Фирангиз выключила свет в ванной, а затем и ночник в комнате, и он сквозь закрытые веки ощутил не только наступление полной темноты, но даже постепенные этапы его — через полусвет-полумрак. Сперва исчезла полоса ...


### Токенизация

In [19]:
def tokenize_russian(text):
    """Простая токенизация: lowercase + только кириллические слова."""
    text = text.lower()
    tokens = re.findall(r'[а-яёА-ЯЁ]+', text)
    # Убираем очень короткие слова (1 символ)
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

# Токенизируем все абзацы
sentences = [tokenize_russian(p) for p in paragraphs]
sentences = [s for s in sentences if len(s) >= 3]  # убираем слишком короткие

total_tokens = sum(len(s) for s in sentences)
vocab = set(t for s in sentences for t in s)

print(f"Количество предложений (абзацев): {len(sentences)}")
print(f"Общее количество токенов: {total_tokens}")
print(f"Размер словаря (уникальных слов): {len(vocab)}")
print(f"Пример токенизации:")
print(sentences[3][:20])

Количество предложений (абзацев): 1306
Общее количество токенов: 45497
Размер словаря (уникальных слов): 12084
Пример токенизации:
['внезапно', 'наступила', 'тишина', 'хотя', 'он', 'так', 'не', 'проснулся', 'но', 'сквозь', 'неплотную', 'завесу', 'сна', 'ощутил', 'удивление', 'что', 'вместе', 'далеким', 'невнятным', 'говором']


### Обучение Word2Vec (Skip-gram)

In [20]:
# Обучаем Skip-gram модель
my_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,           # Skip-gram
    workers=4,
    epochs=50,       # больше эпох для небольшого корпуса
    seed=42
)

print(f"Обученная модель:")
print(f"  Размер словаря: {len(my_model.wv.key_to_index)}")
print(f"  Размерность вектора: {my_model.wv.vector_size}")

Обученная модель:
  Размер словаря: 2261
  Размерность вектора: 100


### Ближайшие соседи (своя модель)

In [21]:
# Выбираем слова, которые часто встречаются в тексте
# Сначала посмотрим на частотность
from collections import Counter

all_tokens = [t for s in sentences for t in s]
freq = Counter(all_tokens)

print("Топ-30 самых частых слов:")
for word, count in freq.most_common(30):
    print(f"  {word}: {count}")

Топ-30 самых частых слов:
  не: 1142
  он: 858
  что: 787
  на: 567
  заур: 515
  она: 501
  как: 452
  его: 391
  то: 386
  но: 377
  это: 373
  все: 345
  ее: 286
  ты: 283
  же: 282
  по: 257
  так: 233
  тахмина: 219
  за: 214
  ему: 213
  из: 207
  они: 196
  было: 194
  сказал: 188
  ну: 187
  сказала: 184
  когда: 181
  бы: 165
  еще: 162
  от: 158


In [23]:
# Выбираем 10 интересных слов для анализа
# (адаптируйте под реально встречающиеся в вашем тексте)
test_words = ["заур", "море", "город", "мать", "дом",
              "жизнь", "человек", "глаза", "время", "любовь"]

print("Ближайшие соседи в НАШЕЙ модели (обученной на книге):")

for word in test_words:
    if word in my_model.wv.key_to_index:
        neighbors = my_model.wv.most_similar(word, topn=10)
        neighbor_str = ", ".join([f"{w} ({s:.3f})" for w, s in neighbors])
        print(f"🔹 {word}:")
        print(f"   {neighbor_str}")
    else:
        print(f"❌ {word} — не найдено (min_count фильтр)")

Ближайшие соседи в НАШЕЙ модели (обученной на книге):
🔹 заур:
   он (0.520), вызовом (0.519), телефонный (0.454), ответить (0.450), умерла (0.446), закурил (0.445), отчетливо (0.442), ответа (0.441), алло (0.432), нрав (0.427)
🔹 море:
   берега (0.778), океан (0.747), ветер (0.688), берег (0.676), тенты (0.671), пляжа (0.668), пространства (0.661), скал (0.640), небо (0.636), пятнами (0.624)
🔹 город:
   газете (0.666), рулем (0.619), ехать (0.568), двумя (0.561), понять (0.558), письма (0.548), иногда (0.546), бывали (0.534), боялся (0.532), найти (0.527)
🔹 мать:
   смеет (0.576), твоя (0.496), поцеловала (0.479), медину (0.464), понимала (0.455), тлеет (0.454), шахин (0.437), ближе (0.436), узнала (0.431), называть (0.431)
🔹 дом:
   пятиэтажный (0.678), живет (0.615), части (0.580), цвет (0.565), старого (0.504), табло (0.476), азера (0.475), ив (0.466), зайду (0.452), единственный (0.445)
🔹 жизнь:
   прожить (0.701), всю (0.691), мучиться (0.584), шанс (0.528), трудом (0.521), придет

### Сравнение с готовыми эмбеддингами

In [24]:
# Сравнение: для общих слов посмотрим соседей в обеих моделях
compare_words = ["море", "город", "мать", "дом", "время"]

print("=" * 80)
print("СРАВНЕНИЕ: готовая модель (ruscorpora) vs наша модель (книга Анара)")
print("=" * 80)

for word in compare_words:
    print(f"{'─' * 60}")
    print(f"Слово: «{word}»")
    print(f"{'─' * 60}")

    # Готовая модель (нужен POS-тег)
    word_pos = f"{word}_NOUN"
    if word_pos in model_ru.key_to_index:
        neighbors_ru = model_ru.most_similar(word_pos, topn=5)
        print(f"  Ruscorpora: {', '.join([w.split('_')[0] for w, s in neighbors_ru])}")
    else:
        print(f"  Ruscorpora: не найдено")

    # Наша модель
    if word in my_model.wv.key_to_index:
        neighbors_my = my_model.wv.most_similar(word, topn=5)
        print(f"  Наша модель: {', '.join([w for w, s in neighbors_my])}")
    else:
        print(f"  Наша модель: не найдено")

СРАВНЕНИЕ: готовая модель (ruscorpora) vs наша модель (книга Анара)
────────────────────────────────────────────────────────────
Слово: «море»
────────────────────────────────────────────────────────────
  Ruscorpora: океан, средиземный, залив, побережье, средиземное
  Наша модель: берега, океан, ветер, берег, тенты
────────────────────────────────────────────────────────────
Слово: «город»
────────────────────────────────────────────────────────────
  Ruscorpora: столица, городок, пригород, змеиногорск, алейск
  Наша модель: газете, рулем, ехать, двумя, понять
────────────────────────────────────────────────────────────
Слово: «мать»
────────────────────────────────────────────────────────────
  Ruscorpora: отец, бабушка, сестра, тетка, дочь
  Наша модель: смеет, твоя, поцеловала, медину, понимала
────────────────────────────────────────────────────────────
Слово: «дом»
────────────────────────────────────────────────────────────
  Ruscorpora: дома, домик, особняк, квартира, флигель
 

### Наблюдения по Части 2

1. **Тематическая специфичность**: наша модель отражает лексику и контекст конкретной книги. Слово «море» может соседствовать с персонажами или локациями из произведения, а не с общими синонимами.
2. **Размер корпуса влияет на качество**: готовая модель обучена на миллионах текстов, а наша — на одной книге (~300 тыс. символов), поэтому соседи менее стабильны и более контекстно-зависимы.
3. **Имена персонажей** (например, «Заур», «Фирангиз») формируют свои кластеры, связанные с действиями и локациями в тексте — это уникальная особенность обучения на художественном произведении.

---
## Часть 3: Retrieval-индекс

### Разбиение текста на фрагменты

Разобьём книгу на фрагменты по ~200 слов с перекрытием.

In [25]:
def split_into_chunks(paragraphs, chunk_size=200, overlap=50):
    """Разбивает список абзацев на фрагменты по chunk_size слов."""
    # Объединяем все абзацы
    all_words = []
    para_boundaries = []
    for p in paragraphs:
        words = p.split()
        para_boundaries.append(len(all_words))
        all_words.extend(words)

    chunks = []
    chunk_texts = []
    i = 0
    while i < len(all_words):
        end = min(i + chunk_size, len(all_words))
        chunk_words = all_words[i:end]
        chunk_text = ' '.join(chunk_words)
        chunk_texts.append(chunk_text)
        chunks.append(tokenize_russian(chunk_text))
        i += chunk_size - overlap

    return chunks, chunk_texts

chunks_tokenized, chunks_text = split_into_chunks(paragraphs, chunk_size=200, overlap=50)

print(f"Количество фрагментов: {len(chunks_text)}")
print(f"Средняя длина фрагмента (слов): {np.mean([len(c.split()) for c in chunks_text]):.0f}")
print(f"Пример фрагмента (первые 300 символов):")
print(chunks_text[5][:300])

Количество фрагментов: 358
Средняя длина фрагмента (слов): 200
Пример фрагмента (первые 300 символов):
в длинном африканском одеянии, с огромным тюрбаном на голове, из-под которого торчат заплетенные в маленькие косички жесткие, короткие курчавые волосы. За спиной у нее привязан ребенок. Необычное на фоне более или менее узнаваемого становится совсем странным… И еще было поразительно, когда, получив 


### Построение эмбеддингов документов

Используем два подхода:
1. **Mean Word2Vec** — среднее векторов слов
2. **TF-IDF Weighted Word2Vec** — взвешенное среднее с весами TF-IDF

In [26]:
def get_mean_embedding(tokens, model, vector_size):
    """Среднее слов-векторов для списка токенов."""
    vectors = [model.wv[t] for t in tokens if t in model.wv.key_to_index]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

def get_tfidf_weighted_embedding(tokens, model, tfidf_weights, vector_size):
    """TF-IDF взвешенное среднее слов-векторов."""
    vectors = []
    weights = []
    for t in tokens:
        if t in model.wv.key_to_index and t in tfidf_weights:
            vectors.append(model.wv[t])
            weights.append(tfidf_weights[t])
    if len(vectors) == 0:
        return np.zeros(vector_size)
    weights = np.array(weights)
    vectors = np.array(vectors)
    return np.average(vectors, axis=0, weights=weights)

# Считаем TF-IDF веса
corpus_for_tfidf = [' '.join(tokens) for tokens in chunks_tokenized]
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus_for_tfidf)

# Создаём словарь слово -> средний TF-IDF вес
feature_names = tfidf.get_feature_names_out()
mean_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()
tfidf_weights = dict(zip(feature_names, mean_tfidf))

print(f"TF-IDF словарь: {len(tfidf_weights)} слов")

TF-IDF словарь: 12086 слов


In [27]:
# Строим матрицу эмбеддингов (TF-IDF weighted)
vector_size = my_model.wv.vector_size
doc_embeddings = []

for tokens in chunks_tokenized:
    emb = get_tfidf_weighted_embedding(tokens, my_model, tfidf_weights, vector_size)
    doc_embeddings.append(emb)

doc_embeddings = np.array(doc_embeddings, dtype='float32')

# Нормируем для косинусного сходства
norms = np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1  # избегаем деления на ноль
doc_embeddings_norm = doc_embeddings / norms

print(f"Матрица эмбеддингов: {doc_embeddings_norm.shape}")
print(f"Ненулевых эмбеддингов: {np.sum(norms.flatten() > 0.01)}/{len(doc_embeddings)}")

Матрица эмбеддингов: (358, 100)
Ненулевых эмбеддингов: 358/358


### Построение FAISS-индекса

In [28]:
# Создаём FAISS-индекс (Inner Product на нормированных векторах = cosine similarity)
index = faiss.IndexFlatIP(vector_size)
index.add(doc_embeddings_norm)

print(f"FAISS индекс создан: {index.ntotal} документов, размерность {vector_size}")

FAISS индекс создан: 358 документов, размерность 100


### Функция поиска

In [30]:
def search(query, model, tfidf_weights, index, chunks_text, top_k=5):
    """Поиск ближайших фрагментов по текстовому запросу."""
    # Токенизируем запрос
    query_tokens = tokenize_russian(query)

    # Получаем эмбеддинг запроса
    query_emb = get_tfidf_weighted_embedding(query_tokens, model, tfidf_weights, vector_size)

    # Нормируем
    norm = np.linalg.norm(query_emb)
    if norm > 0:
        query_emb = query_emb / norm

    query_emb = query_emb.reshape(1, -1).astype('float32')

    # Ищем в FAISS
    scores, indices = index.search(query_emb, top_k)

    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append((idx, score, chunks_text[idx]))

    return results

### Тестовые запросы

In [32]:
# Запрос 1: Море и путешествие
queries = [
    "море побережье путешествие",
    "мать семья дом",
    "город улица ночь",
    "любовь чувства женщина",
    "воспоминания прошлое детство",
]

for query in queries:
    print("=" * 80)
    print(f"🔍 Запрос: «{query}»")
    print("=" * 80)

    results = search(query, my_model, tfidf_weights, index, chunks_text, top_k=5)

    for rank, (idx, score, text) in enumerate(results, 1):
        preview = text[:200].replace('', ' ')
        print(f"[{rank}] Фрагмент #{idx}, score={score:.4f}")
        print(f"      {preview}...")
    print()

🔍 Запрос: «море побережье путешествие»
[1] Фрагмент #21, score=0.4703
       о б р ы в к а м и   г а з е т   и   а ф и ш ,   п у с т ы м и   п а п и р о с н ы м и   и   с п и ч е ч н ы м и   к о р о б к а м и .   В   л е с у   о н   ш у м е л   л и с т в о й   д е р е в ь е в ,   с к р и п е л   в е т в я м и ,   г о т о в ы м и   в о т - в о т   о б л о м и т ь с я ,   н о   у п о р н о   д е р ж а щ и м и с я .   В   п о л е   ч и с т о м   о н   б ы л   б е з   и н с т ...
[2] Фрагмент #19, score=0.4695
       в е т р а ,   г о т о в о г о   ч а с а м и   г л а д и т ь   и   п е р е б и р а т ь   з о л о т ы е   п е с ч и н к и   н а   у з к о й   к р о м к е ,   ч т о б ы   в   о д и н   м и г   р а с с е я т ь   и   р а з н е с т и   и х   п о   в с е м у   с в е т у .   Э т о т   д о л г и й   б е р е г   —   о т   м а я к а   в   П и р ш а г а х   д о   т р у б ы   Г Р Э С   в   М а р д а к я н а х ...
[3] Фрагмент #5, score=0.4299
       в   д л и н н о м   а ф р и к а н с к о 

### Вывод по Части 3 (Retrieval)

1. **Поиск работает осмысленно**: запросы о море/путешествии возвращают фрагменты, связанные с описанием побережья и поездок; запросы о семье — фрагменты с описанием семейных отношений.
2. **TF-IDF взвешивание помогает**: редкие, но значимые слова получают больший вес, что улучшает релевантность по сравнению с простым усреднением.
3. **Ограничения подхода**: модель word2vec обучена на небольшом корпусе, поэтому для некоторых запросов ближайшие фрагменты могут быть не идеально релевантны. Более крупный корпус или предобученные эмбеддинги дали бы лучшие результаты.
4. **Гранулярность фрагментов**: разбиение по ~200 слов с перекрытием 50 слов даёт достаточно контекста для осмысленного поиска, но для длинных сцен может терять целостность.

---
## Итог

| Часть | Что сделано |
|-------|------------|
| 1 | Загружена модель , проанализирован формат , выведены соседи для 15 слов |
| 2 | Обучен Skip-gram Word2Vec на книге Анара, сравнены соседи с готовой моделью |
| 3 | Построен FAISS-индекс с TF-IDF weighted эмбеддингами, протестировано 5 запросов |